In [ ]:
#import pandas as pd

#df_TANG = pd.read_csv(r"/work/DISCOURSE/Data/Discourse/prosodic_features_TANG_repo_style_LOGF0_CLEANED.csv")
#df_UWO = pd.read_csv(r"/work/DISCOURSE/Data/Discourse/prosodic_features_UWO_repo_style_LOGF0_CLEANED.csv")
#df_TOPSY = pd.read_csv(r"/work/DISCOURSE/Data/Discourse/prosodic_features_TOPSY_repo_style_LOGF0_CLEANED.csv")
#df_DAIC = pd.read_csv(r"/work/DISCOURSE/Data/DAIC-WOZ/prosodic_features_DAIC_repo_style_LOGF0_CLEANED.csv")

#df_ALL = pd.concat([df_TANG, df_UWO, df_TOPSY, df_DAIC], ignore_index=True)

#out_path = r"./Data/prosodic_features_ALL_CLEANED.csv"
#df_ALL.to_csv(out_path, index=False)

#print(df_ALL.shape)
#print(df_ALL.columns.tolist())

In [ ]:
"""
Code for converting word-row based dataframes created during prosodic feature extraction into utterance-based .pkl datasets, enabling model regression

Run cell above above to concatenate multiple dataframes
"""

import os
import pickle
import pandas as pd

# pathing
feature_csv = r"./Data/prosodic_features_ALL_CLEANED.csv"
output_base_dir = r"./Data/prosodic_features_splits_CLEANED/"
os.makedirs(output_base_dir, exist_ok=True)

df = pd.read_csv(feature_csv, low_memory=False)

if "psychosis_remission" not in df.columns and "psychosis remission" in df.columns:
    df["psychosis_remission"] = df["psychosis remission"]

for col in [
    "split",
    "utterance_id",
    "speaker",
    "label_depression",
    "label_psychosis",
    "word_text",
    "psychosis_remission",
]:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

for col in ["label_depression", "label_psychosis", "psychosis_remission"]:
    if col in df.columns:
        df[col] = df[col].replace(
            {"nan": pd.NA, "None": pd.NA, "": pd.NA, "NA": pd.NA, "<NA>": pd.NA}
        )

# feature mapping to columns within dataframe
feature_mapping = {
    "relative_prominence": "prom_rel",
    "absolute_prominence": "prom_abs",
    "pitch": ["f0_dct_0_z", "f0_dct_1_z", "f0_dct_2_z", "f0_dct_3_z"],
    "loudness": "intensity_z",
    "duration": "duration_per_syllable",
    "pause": "pause",
}

split_mapping = {
    "train": ["train", "train-clean", "train_clean", "training"],
    "val": ["val", "valid", "validation", "dev"],
    "test": ["test", "eval", "evaluation", "test-clean"],
}

group_keys = ["speaker", "utterance_id"]

# helpers
def save_pickle(path, texts, labels):
    with open(path, "wb") as f:
        pickle.dump({"texts": texts, "labels": labels}, f)


def norm_label(x):
    if x is pd.NA or pd.isna(x):
        return None
    s = str(x).strip().upper()
    return None if s in {"", "NAN", "NONE", "NA", "<NA>"} else s


def is_healthy(dep_label, psy_label):
    dep = norm_label(dep_label)
    psy = norm_label(psy_label)
    dep_ok = dep is None or dep == "H"
    psy_ok = psy is None or psy == "N"
    return dep_ok and psy_ok


def is_depressed(dep_label):
    return norm_label(dep_label) == "D"


def is_psychotic(psy_label):
    return norm_label(psy_label) == "Y"


def has_psychosis_remission(value, target):
    return norm_label(value) == str(target).strip().upper()


def is_vector_feature(col_spec):
    return isinstance(col_spec, (list, tuple))


def get_feature_group(group, col_spec):
    group = group.sort_values("word_index")

    if is_vector_feature(col_spec):
        needed_cols = list(col_spec)
        group = group.dropna(subset=needed_cols)

        if len(group) == 0:
            return [], []

        return (
            group["word_text"].astype(str).tolist(),
            group[needed_cols].astype(float).values.tolist(),
        )

    group = group.dropna(subset=[col_spec])

    if len(group) == 0:
        return [], []

    return (
        group["word_text"].astype(str).tolist(),
        group[col_spec].astype(float).tolist(),
    )


def init_breakdown():
    return {
        "healthy_texts": [],
        "healthy_labels": [],
        "depression_texts": [],
        "depression_labels": [],
        "psychosis_texts": [],
        "psychosis_labels": [],
        "remission_y_texts": [],
        "remission_y_labels": [],
        "remission_n_texts": [],
        "remission_n_labels": [],
    }


def add_to_breakdown(store, text, label_values, dep_label, psy_label, psy_rem):
    if is_healthy(dep_label, psy_label):
        store["healthy_texts"].append(text)
        store["healthy_labels"].append(label_values)

    if is_depressed(dep_label):
        store["depression_texts"].append(text)
        store["depression_labels"].append(label_values)

    if is_psychotic(psy_label):
        store["psychosis_texts"].append(text)
        store["psychosis_labels"].append(label_values)

    if has_psychosis_remission(psy_rem, "Y"):
        store["remission_y_texts"].append(text)
        store["remission_y_labels"].append(label_values)

    if has_psychosis_remission(psy_rem, "N"):
        store["remission_n_texts"].append(text)
        store["remission_n_labels"].append(label_values)


def print_breakdown(split_name, all_texts, store):
    print(f"\n--- {split_name.upper()} split breakdown ---")
    print(f"Total utterances:                {len(all_texts)}")
    print(f"Healthy:                         {len(store['healthy_texts'])}")
    print(f"Depressed:                       {len(store['depression_texts'])}")
    print(f"Psychotic:                       {len(store['psychosis_texts'])}")
    print(f"Psychosis remission = Y:         {len(store['remission_y_texts'])}")
    print(f"Psychosis remission = N:         {len(store['remission_n_texts'])}")


# main
for friendly_name, col_spec in feature_mapping.items():
    output_dir = os.path.join(output_base_dir, friendly_name)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\nProcessing feature: {friendly_name} ({col_spec})")

    if is_vector_feature(col_spec):
        total_rows_with_feature = int(df[list(col_spec)].notna().all(axis=1).sum())
    else:
        total_rows_with_feature = int(df[col_spec].notna().sum())

    print(f"Rows with usable labels: {total_rows_with_feature}/{len(df)}")

    for target_split, source_split_names in split_mapping.items():
        split_df = df[df["split"].isin(source_split_names)].copy()

        all_texts, all_labels = [], []
        breakdown = init_breakdown()

        total_groups = 0
        skipped_empty = 0
        skipped_length_mismatch = 0

        for group_id, group in split_df.groupby(group_keys):
            total_groups += 1

            text_tokens, label_values = get_feature_group(group, col_spec)

            if len(text_tokens) == 0 or len(label_values) == 0:
                skipped_empty += 1
                continue

            if len(text_tokens) != len(label_values):
                skipped_length_mismatch += 1
                print(
                    f"Skipping utterance {group_id}: "
                    f"{len(text_tokens)} words but {len(label_values)} labels"
                )
                continue

            text = " ".join(text_tokens)

            all_texts.append(text)
            all_labels.append(label_values)

            dep_label = (
                group["label_depression"].iloc[0]
                if "label_depression" in group.columns
                else pd.NA
            )
            psy_label = (
                group["label_psychosis"].iloc[0]
                if "label_psychosis" in group.columns
                else pd.NA
            )
            psy_rem = (
                group["psychosis_remission"].iloc[0]
                if "psychosis_remission" in group.columns
                else pd.NA
            )

            add_to_breakdown(
                breakdown,
                text,
                label_values,
                dep_label,
                psy_label,
                psy_rem,
            )

        # save main split
        out_file = os.path.join(output_dir, f"{target_split}.pkl")
        save_pickle(out_file, all_texts, all_labels)

        print(f"Saved {target_split} split to {out_file} with {len(all_texts)} utterances")
        print_breakdown(target_split, all_texts, breakdown)

        # save subgroup pkl-files for every distinct test-split
        subgroup_files = {
            f"{target_split}_healthy.pkl": (
                breakdown["healthy_texts"],
                breakdown["healthy_labels"],
            ),
            f"{target_split}_depression.pkl": (
                breakdown["depression_texts"],
                breakdown["depression_labels"],
            ),
            f"{target_split}_psychosis.pkl": (
                breakdown["psychosis_texts"],
                breakdown["psychosis_labels"],
            ),
            f"{target_split}_psychosis_remission_Y.pkl": (
                breakdown["remission_y_texts"],
                breakdown["remission_y_labels"],
            ),
            f"{target_split}_psychosis_remission_N.pkl": (
                breakdown["remission_n_texts"],
                breakdown["remission_n_labels"],
            ),
        }

        for filename, (texts, labels) in subgroup_files.items():
            save_pickle(os.path.join(output_dir, filename), texts, labels)

        # Backward-compatible test filenames
        if target_split == "test":
            save_pickle(
                os.path.join(output_dir, "test_healthy.pkl"),
                breakdown["healthy_texts"],
                breakdown["healthy_labels"],
            )
            save_pickle(
                os.path.join(output_dir, "test_depressed.pkl"),
                breakdown["depression_texts"],
                breakdown["depression_labels"],
            )
            save_pickle(
                os.path.join(output_dir, "test_psychosis.pkl"),
                breakdown["psychosis_texts"],
                breakdown["psychosis_labels"],
            )
            save_pickle(
                os.path.join(output_dir, "test_psychosis_remission_Y.pkl"),
                breakdown["remission_y_texts"],
                breakdown["remission_y_labels"],
            )
            save_pickle(
                os.path.join(output_dir, "test_psychosis_remission_N.pkl"),
                breakdown["remission_n_texts"],
                breakdown["remission_n_labels"],
            )

        print("\nSaved subgroup files:")
        print(f"  Healthy                   -> {os.path.join(output_dir, f'{target_split}_healthy.pkl')}")
        print(f"  Depression                -> {os.path.join(output_dir, f'{target_split}_depression.pkl')}")
        print(f"  Psychosis                 -> {os.path.join(output_dir, f'{target_split}_psychosis.pkl')}")
        print(f"  Psychosis remission = Y   -> {os.path.join(output_dir, f'{target_split}_psychosis_remission_Y.pkl')}")
        print(f"  Psychosis remission = N   -> {os.path.join(output_dir, f'{target_split}_psychosis_remission_N.pkl')}")

        print(
            f"  Total grouped utterances: {total_groups} | "
            f"Empty after feature filtering: {skipped_empty} | "
            f"Length mismatches: {skipped_length_mismatch}"
        )

print("\nAll feature pickle files generated successfully.")

In [1]:
"""
This cell then generates yaml-files for data and task configuration and stores these in the correct folders within contextual-redundancy.
These all rely on the custom_pickle_datamodule.py in src/data/ to work. Within these configs you assign loss outputs, and path to your data.
You assign for each feature, what regressor they should call - the ones in this cell are for contextual models.
Model finetuning hyperparameters are also assigned. You can use the hparams lightning function to estimate some of these
"""

import os
import yaml

repo_root = os.getcwd()
splits_base_dir = f"{repo_root}/Data/prosodic_features_splits_CLEANED"

experiment_dir = os.path.join(repo_root, "configs", "experiment", "emnlp", "finetuning")
datamodule_dir = os.path.join(repo_root, "configs", "data")
model_task_dir = os.path.join(repo_root, "configs", "model_task")

os.makedirs(experiment_dir, exist_ok=True)
os.makedirs(datamodule_dir, exist_ok=True)
os.makedirs(model_task_dir, exist_ok=True)

feature_mapping = {
    "relative_prominence": "prom_rel",
    "absolute_prominence": "prom_abs",
    "pitch": ["f0_dct_0_z", "f0_dct_1_z", "f0_dct_2_z", "f0_dct_3_z"],
    "loudness": "intensity_z",
    "duration": "duration_per_syllable",
    "pause": "pause",
}

model_task_by_feature = {
    "relative_prominence": "token_tagging_regressor_mle_relative_cm.yaml",
    "absolute_prominence": "token_tagging_regressor_prominence_cm.yaml",
    "pitch": "token_tagging_vector_regressor_mle_cm.yaml",
    "loudness": "token_tagging_regressor_mle_energy_cm.yaml",
    "duration": "token_tagging_regressor_mle_duration_cm.yaml",
    "pause": "token_tagging_regressor_pause_mle_cm.yaml",
}

model_name_by_feature = {
    "pitch": "bert-base-cased",
    "relative_prominence": "bert-large-cased",
    "absolute_prominence": "bert-large-cased",
    "loudness": "bert-large-cased",
    "duration": "bert-large-cased",
    "pause": "bert-large-cased",
}

num_labels_by_feature = {
    "relative_prominence": 1,
    "absolute_prominence": 1,
    "pitch": 4,
    "loudness": 1,
    "duration": 1,
    "pause": 1,
}

loss_dir_by_feature = {
    "relative_prominence": "./losses_cm_CLEAN/prom_rel",
    "absolute_prominence": "./losses_cm_CLEAN/prom_abs",
    "pitch": "./losses_cm_CLEAN/f0_dct",
    "loudness": "./losses_cm_CLEAN/intensity",
    "duration": "./losses_cm_CLEAN/duration",
    "pause": "./losses_cm_CLEAN/pause",
}

lr_by_feature = {
    "relative_prominence": 0.000007,
    "absolute_prominence": 0.000009,
    "pitch": 0.000003,
    "loudness": 0.000004,
    "duration": 0.000005,
    "pause": 0.000003,
}

w_decay_by_feature = {
    "relative_prominence": 0.0001,
    "absolute_prominence": 0.0001,
    "pitch": 0.002,
    "loudness": 0.0004,
    "duration": 0.0008,
    "pause": 0.002,
}

batch_size_by_feature = {
    "relative_prominence": 16,
    "absolute_prominence": 16,
    "pitch": 8,
    "loudness": 16,
    "duration": 4,
    "pause": 8,
}

accumulate_grad_batches_by_feature = {k: 1 for k in feature_mapping}
dropout_by_feature = {k: 0.1 for k in feature_mapping}

for feature in feature_mapping:
    feature_dir = os.path.join(splits_base_dir, feature)

    model_name = model_name_by_feature[feature]


    experiment_cfg = {
        "defaults": [
            {"override /data": f"{feature}_datamodule.yaml"},
            {"override /model_task": model_task_by_feature[feature]},
            {"override /callbacks": "prominence_regression.yaml"},
            {"override /logger": "many_loggers.yaml"},
            {"override /trainer": "gpu.yaml"},
            {"override /hydra/job_logging": "default"},
            {"override /hydra/hydra_logging": "default"},
        ],
        "tags": [model_name, "prosody", "regression", feature],
        "seed": 12345,
        "trainer": {
            "min_epochs": 1,
            "max_epochs": 50,
            "gradient_clip_val": 1,
            "precision": "16-mixed",
            "accumulate_grad_batches": accumulate_grad_batches_by_feature[feature],
        },
        "model_task": {
            "huggingface_model": model_name,
            "num_labels": num_labels_by_feature[feature],
            "freeze_lm": False,
            #"train_last_k_layers": 2,
            "use_mlp": False,
            "p_dropout": dropout_by_feature[feature],
            "loss_dir": loss_dir_by_feature[feature],
            "optimizer": {
                "lr": lr_by_feature[feature],
                "weight_decay": w_decay_by_feature[feature],
            },
            "scheduler": {
                "patience": 2,
            },
        },
        "callbacks": {
            "early_stopping": {
                "patience": 4,
            }
        },
    }

    experiment_path = os.path.join(experiment_dir, f"{feature}_finetuning.yaml")
    with open(experiment_path, "w", encoding="utf-8") as f:
        f.write("# @package _global_\n")
        yaml.dump(
            experiment_cfg,
            f,
            sort_keys=False,
            default_flow_style=False,
            allow_unicode=True,
        )
    print("Saved:", experiment_path)

    datamodule_cfg = {
        "_target_": "src.data.custom_pickle_datamodule.CustomPickleDataModule",
        "data_dir": feature_dir,
        "train_file": "train.pkl",
        "val_file": "val.pkl",
        "test_file": "test.pkl",
        "batch_size": batch_size_by_feature[feature],
        "model_name": model_name,
        "score_last_token": True,
        "relative_to_mean": False,
        "word_stats_path": None,
        "num_workers": 0,
        "pin_memory": True,
        "max_context_words": 10,
        "windows_per_utterance": 4,
    }

    datamodule_path = os.path.join(datamodule_dir, f"{feature}_datamodule.yaml")
    with open(datamodule_path, "w", encoding="utf-8") as f:
        yaml.dump(
            datamodule_cfg,
            f,
            sort_keys=False,
            default_flow_style=False,
            allow_unicode=True,
        )
    print("Saved:", datamodule_path)

print("All configs generated successfully.")

Saved: d:\thesis_data\ucloud-contextual-redundancy\contextual-redundancy\configs\experiment\emnlp\finetuning\relative_prominence_finetuning.yaml
Saved: d:\thesis_data\ucloud-contextual-redundancy\contextual-redundancy\configs\data\relative_prominence_datamodule.yaml
Saved: d:\thesis_data\ucloud-contextual-redundancy\contextual-redundancy\configs\experiment\emnlp\finetuning\absolute_prominence_finetuning.yaml
Saved: d:\thesis_data\ucloud-contextual-redundancy\contextual-redundancy\configs\data\absolute_prominence_datamodule.yaml
Saved: d:\thesis_data\ucloud-contextual-redundancy\contextual-redundancy\configs\experiment\emnlp\finetuning\pitch_finetuning.yaml
Saved: d:\thesis_data\ucloud-contextual-redundancy\contextual-redundancy\configs\data\pitch_datamodule.yaml
Saved: d:\thesis_data\ucloud-contextual-redundancy\contextual-redundancy\configs\experiment\emnlp\finetuning\loudness_finetuning.yaml
Saved: d:\thesis_data\ucloud-contextual-redundancy\contextual-redundancy\configs\data\loudnes

In [ ]:
# cell to just check if pkl formatting looks correct

import pickle, numpy as np

p = "./Data/prosodic_features_splits_CLEANED/duration/train.pkl"
with open(p, "rb") as f:
    d = pickle.load(f)

print(d.keys())
print(len(d["texts"]), len(d["labels"]))
print(d["texts"][0])
print(np.asarray(d["labels"][0]).shape)
print(d["labels"][0][1])